# Sensitivitätsanalyse Patagonien

In [ ]:
import os
import papermill as pm
import pandas as pd
from scipy.stats import triang

# ==========================================
# 🎛️ Anzahl Monte-Carlo-Runs festlegen
num_runs = 200 # <== Hier kannst du auf 10, 100, 500 usw. erhöhen
# ==========================================

# 📥 Verteilungsdefinition aus Excel einlesen
df = pd.read_excel("Sensitivitäten.xlsx", sheet_name="Datenauswahl", decimal=',')
df.set_index("Variable", inplace=True)

# 📄 Struktur für Monte Carlo-Ergebnis laden
df_mc = pd.read_excel("Sensitivitäten.xlsx", sheet_name="MonteCarlo", engine="openpyxl")

# 📁 Modellpfade
model_dir = os.path.abspath(os.path.join("..", "Ammoniak_Greenfield_Patagonien"))
notebook_path = os.path.join(model_dir, "Modell_Ammoniak_Greenfield_Patagonien.ipynb")
trash_path = "trash_output.ipynb"

# 🔁 Hauptloop für die Runs
for i in range(1, num_runs + 1):
    run_name = f"Run_{i}"
    print(f"➡️ Starte {run_name} ...")

    # ▪️ Neue Ziehung
    overrides = {}
    for var in df.index:
        low = df.loc[var, "lower bound"]
        mode = df.loc[var, "Wert"]
        high = df.loc[var, "upper bound"]
        c = (mode - low) / (high - low)
        dist = triang(c=c, loc=low, scale=high - low)
        overrides[var] = round(dist.rvs(), 2)

    # ▪️ Notebook ausführen mit overrides
    pm.execute_notebook(
        notebook_path,
        output_path=trash_path,
        parameters={"overrides": overrides},
        cwd=model_dir
    )
    os.remove(trash_path)

    # ▪️ Ergebnis einlesen
    ergebnisse_path = os.path.join(model_dir, "Ergebnisse.csv")
    Ergebnisse_df = pd.read_csv(ergebnisse_path, sep=';', decimal=',')
    Ergebnisse_df.set_index("Variable", inplace=True)
    Ergebnisse = Ergebnisse_df["Wert"].to_dict()

    # ▪️ Neue Spalte in df_mc anlegen
    df_mc[run_name] = None

    # ▪️ Inputs eintragen
    for var, value in overrides.items():
        df_mc.loc[df_mc["Variable"] == var, run_name] = value

    # ▪️ Outputs eintragen
    for var, value in Ergebnisse.items():
        df_mc.loc[df_mc["Variable"] == var, run_name] = value

    # ▪️ Zahlen erzwingen
    df_mc[run_name] = pd.to_numeric(df_mc[run_name], errors="coerce")

    print(f"✅ {run_name} abgeschlossen.")

# 💾 Finale Ausgabe
df_mc.to_csv("MonteCarlo_Ergebnisse.csv", sep=";", encoding="utf-8-sig", decimal=',', index=False)
print("🧾 Export abgeschlossen → MonteCarlo_Ergebnisse.csv")


# Darstellung der Ergebnisse

In [ ]:
import matplotlib.pyplot as plt

col_n2 = col_asu = "#e5742e"
col_pv = "#e4cb3a"
col_wka = "#128fcf"
col_h2 = col_pem = "#14689b"
col_nh3 = col_hb = "#822181"
col_h2s = "#79CADD"
col_bat = "#8CBF3E"
col_buy = col_chem = "#1D9838"
col_etc = "#828282"
col_el = col_sell = "#E02229"

# Farben und Variablen
colors = [col_nh3, col_n2, col_h2, col_pem, col_h2s, col_el, col_bat, col_wka, col_pv]
variables = ["calc_lcof_hb", "calc_lcon_asu", "calc_lcoh_h2", "calc_lcoh_pem", "calc_lcoh_h2s", "calc_lcoe_mix", "calc_lcoe_bat", "calc_lcoe_wka", "calc_lcoe_pv"]
labels = ["LCO Ammonia [€/kg_NH3]", "LCO Nitrogren [€/kg_N2]", "LCO Hydrogen [€/kg_H2]", "LCO Electrolysis [€/kg_H2]", "LCO Hydrogen Storage [€/kg_H2]", "LCO Electricity mix [ct/kWh]", "LCO Battery [ct/kWh]", "LCO Wind [ct/kWh]", "LCO PV [ct/kWh]"]

# Datei einlesen
df = pd.read_csv("MonteCarlo_Ergebnisse.csv", sep=";", decimal=",")
df.set_index("Variable", inplace=True)

# Relevante Daten extrahieren und transponieren
df_plot = df.loc[variables].drop(columns=["Prozess", "Einheit"], errors="ignore").T


for var in ["calc_lcof_hb"]:
    df_plot[var] /= 1000

# Plot vorbereiten
fig, ax = plt.subplots(figsize=(7, 5))

cut = 5.5
ax.axhspan(cut, 10, facecolor=col_etc, alpha=0.1)
ax.axhspan(-0.75, cut, facecolor=col_etc, alpha=0.3)
ax.set_ylim([0.25, 9.75])
ax.text(20, 5.75, "Electricity supply", fontsize=9, color="black")
ax.text(20, 5.1, "Conversion chain", fontsize=9, color="black")

# Boxplots zeichnen
box = ax.boxplot(
    [df_plot[var].dropna() for var in variables],
    vert=False,
    patch_artist=True,
    labels=labels
)

# Farben zuweisen
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_edgecolor('black')
for whisker in box['whiskers']:
    whisker.set_color('black')
for cap in box['caps']:
    cap.set_color('black')
for median in box['medians']:
    median.set_color('black')

# Stil
ax.set_xlabel("Levelized cost in € per functional unit", fontsize=10)
ax.set_title("Monte Carlo: Levelized Cost of Technology (LCOT)", fontsize=10)
ax.grid(which='major', linestyle='--', linewidth=0.75, color='lightgray')
ax.set_axisbelow(True)
plt.xlim(0, 35)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

plt.tight_layout()
plt.savefig("Bilder/calc_lcot_boxplot.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# calc_GuV-Werte extrahieren und konvertieren
guv_values = df.loc["calc_GuV"].drop(labels=["Prozess", "Einheit"], errors="ignore").astype(float)

# Plot
fig, ax = plt.subplots(figsize=(7, 5))

box = ax.boxplot(
    [guv_values.dropna()],
    vert=False,
    patch_artist=True,
    labels=["Annual Profit [€/a]"]
)

# Farbe & Stil
for patch in box['boxes']:
    patch.set_facecolor(col_buy)
    patch.set_edgecolor('black')
for element in ['whiskers', 'caps', 'medians']:
    for line in box[element]:
        line.set_color('black')

# Grid & Layout
ax.set_xlabel("Net Revenue per Year [€]", fontsize=10)
ax.set_title("Monte Carlo: Annual Profit Distribution", fontsize=10)
ax.grid(which='major', linestyle='--', linewidth=0.75, color='lightgray')
ax.set_axisbelow(True)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

plt.tight_layout()
plt.savefig("Bilder/calc_guv_boxplot.png", dpi=300, bbox_inches='tight')
plt.show()

# Analyse der Ergebnisse

In [ ]:
import seaborn as sns
# df_mc vorbereiten
df_mc["Prozess"] = df_mc["Prozess"].fillna(method="ffill")

# Inputs und Outputs
input_vars = df_mc[df_mc["Prozess"] == "Inputs"]["Variable"]
output_vars = df_mc[df_mc["Prozess"] == "Outputs"]["Variable"]

# Daten extrahieren
input_data = df_mc[df_mc["Variable"].isin(input_vars)] \
    .set_index("Variable") \
    .drop(columns=["Prozess", "Einheit"], errors="ignore") \
    .T

output_data = df_mc[df_mc["Variable"].isin(output_vars)] \
    .set_index("Variable") \
    .drop(columns=["Prozess", "Einheit"], errors="ignore") \
    .T

# sicherstellen, dass alles float ist
input_data = input_data.apply(pd.to_numeric, errors="coerce")
output_data = output_data.apply(pd.to_numeric, errors="coerce")

# Korrelationsmatrix berechnen
corr_matrix = input_data.corrwith(output_data, axis=0).T  # Input ↔ Outputs
heatmap_df = pd.DataFrame(index=input_data.columns, columns=output_data.columns)

for out in output_data.columns:
    heatmap_df[out] = input_data.corrwith(output_data[out])

# Plot
plt.figure(figsize=(10, 6))
sns.set(style="whitegrid")

ax = sns.heatmap(
    heatmap_df.astype(float).round(2),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    linecolor="gray",
    cbar_kws={'label': 'Pearson Correlation'}
)

plt.title("Correlation between Inputs and Outputs", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("Bilder/sensitivity_correlation_matrix.png", dpi=300, bbox_inches='tight')
plt.show()